# 03 · Colab timing check (free CPU runtime)

Spec: the full pipeline must process the test set in under 10 minutes on a free Google Colab CPU. This notebook takes the code from the public Hugging Face Space (same `src/` as the repository), installs the pinned versions, downloads the dataset from Mendeley (SHA-256 checked) and times the frozen final pipeline on all 137 test images. No parameter is changed.

In [ ]:
!git clone -q https://huggingface.co/spaces/marwan19-saber/crack-measurement cm
!pip -q install numpy==2.5.4 opencv-python-headless==5.0.0.93 scipy==1.18.1 scikit-image==0.26.0 PyYAML==6.0.3
!cd cm && python -c "import sys, numpy, cv2, skimage; print(sys.version.split()[0], numpy.__version__, cv2.__version__, skimage.__version__)"
!nproc; lscpu | grep 'Model name'; free -g | head -2

In [ ]:
!which unrar || apt-get -qq install -y unrar > /dev/null
!cd cm && python data/download_dataset.py --out data/raw

In [ ]:
%%writefile cm/time_test.py
import time
from src import data, metrics, pipeline
config = pipeline.load_config('configs/final_pipeline.yaml')
pairs = {img.stem: (img, gt) for img, gt in data.list_pairs('data/raw/concrete_crack')}
names = data.load_split('data/split.json')['test']
start, ious = time.time(), []
for n in names:
    gray = data.load_image(pairs[n][0], size=1008)
    gt = data.load_ground_truth(pairs[n][1], size=1008)
    ious.append(metrics.iou(pipeline.run(gray, config)[0], gt))
total = time.time() - start
print(f'{len(names)} test images | mean IoU {sum(ious) / len(ious):.4f} | '
      f'total {total:.1f} s = {total / 60:.2f} min | {total / len(names):.2f} s per image')

In [ ]:
!cd cm && python time_test.py